# Invoice Field Extraction with Donut Transformer

This notebook extracts structured invoice fields using **Donut** (Document Understanding Transformer),
an OCR-free, end-to-end vision-language model.

**Architecture:** Donut combines a **Swin Transformer** vision encoder with a **BART** text decoder.
The image is processed directly by the encoder — no external OCR step is needed — and the decoder
generates structured text (JSON tokens or natural-language answers) conditioned on a task prompt.

**Two complementary models are used:**

| Model | Purpose | Covers |
|---|---|---|
| `to-be/donut-base-finetuned-invoices` | Invoice-header extraction (single forward pass → JSON) | `invoice_number`, `invoice_date`, `total_amount` |
| `naver-clova-ix/donut-base-finetuned-docvqa` | Document question answering | `issuer_name`, `recipient_name`, `due_date` |

The final output for each invoice image is a 6-field dictionary identical in structure to the
regex-based extractor, enabling direct comparison.

## 1) Install dependencies and imports

In [ ]:
import subprocess, sys, re as _re

def _ensure_installed(*packages: str) -> None:
    for pkg in packages:
        try:
            __import__(_re.split(r"[<>=!]", pkg)[0])
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_ensure_installed("transformers<5", "torch", "sentencepiece", "protobuf", "Pillow")

from __future__ import annotations

import csv
import re
import time
from collections import defaultdict
from pathlib import Path

import torch
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PROJECT_ROOT = Path.cwd()
IMAGES_DIR = PROJECT_ROOT / "images"

print(f"Device : {DEVICE}")
print(f"Project: {PROJECT_ROOT}")

## 2) Load models

In [ ]:
INVOICE_MODEL_ID = "to-be/donut-base-finetuned-invoices"
DOCVQA_MODEL_ID = "naver-clova-ix/donut-base-finetuned-docvqa"

print("Loading invoice model …")
inv_processor = DonutProcessor.from_pretrained(INVOICE_MODEL_ID)
inv_model = VisionEncoderDecoderModel.from_pretrained(INVOICE_MODEL_ID).to(DEVICE)
inv_model.eval()
print("  ✓ invoice model loaded")

print("Loading DocVQA model …")
vqa_processor = DonutProcessor.from_pretrained(DOCVQA_MODEL_ID)
vqa_model = VisionEncoderDecoderModel.from_pretrained(DOCVQA_MODEL_ID).to(DEVICE)
vqa_model.eval()
print("  ✓ DocVQA model loaded")

## 3) Extraction functions

In [ ]:
def extract_with_invoice_model(image: Image.Image) -> dict:
    """Run the invoice-finetuned Donut model and return parsed JSON fields."""
    task_prompt = "<s_cord-v2>"
    decoder_input_ids = inv_processor.tokenizer(
        task_prompt, add_special_tokens=False, return_tensors="pt"
    ).input_ids.to(DEVICE)

    pixel_values = inv_processor(image, return_tensors="pt").pixel_values.to(DEVICE)

    with torch.no_grad():
        outputs = inv_model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=inv_model.decoder.config.max_position_embeddings,
            pad_token_id=inv_processor.tokenizer.pad_token_id,
            eos_token_id=inv_processor.tokenizer.eos_token_id,
            early_stopping=True,
            bad_words_ids=[[inv_processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )

    sequence = inv_processor.batch_decode(outputs.sequences)[0]
    sequence = sequence.replace(inv_processor.tokenizer.eos_token, "").replace(
        inv_processor.tokenizer.pad_token, ""
    )
    parsed = inv_processor.token2json(sequence)
    return parsed


def extract_with_docvqa(image: Image.Image, question: str) -> str | None:
    """Ask a single question about the document image using the DocVQA model."""
    task_prompt = "<s_docvqa><s_question>{question}</s_question><s_answer>"
    prompt = task_prompt.replace("{question}", question)

    decoder_input_ids = vqa_processor.tokenizer(
        prompt, add_special_tokens=False, return_tensors="pt"
    ).input_ids.to(DEVICE)

    pixel_values = vqa_processor(image, return_tensors="pt").pixel_values.to(DEVICE)

    with torch.no_grad():
        outputs = vqa_model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=vqa_model.decoder.config.max_position_embeddings,
            pad_token_id=vqa_processor.tokenizer.pad_token_id,
            eos_token_id=vqa_processor.tokenizer.eos_token_id,
            early_stopping=True,
            bad_words_ids=[[vqa_processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )

    sequence = vqa_processor.batch_decode(outputs.sequences)[0]
    sequence = sequence.replace(vqa_processor.tokenizer.eos_token, "").replace(
        vqa_processor.tokenizer.pad_token, ""
    )
    parsed = vqa_processor.token2json(sequence)

    answer = None
    if isinstance(parsed, dict):
        answer = parsed.get("answer") or parsed.get("text_sequence")
    elif isinstance(parsed, list) and parsed:
        first = parsed[0] if isinstance(parsed[0], dict) else {}
        answer = first.get("answer") or first.get("text_sequence")

    if answer and isinstance(answer, str):
        answer = answer.strip()
        if not answer:
            return None
    return answer


# ── field‐mapping helpers ──────────────────────────────────────────────

_INV_FIELD_MAP = {
    "InvoiceNumber": "invoice_number",
    "invoicenumber": "invoice_number",
    "invoice_number": "invoice_number",
    "DocumentDate": "invoice_date",
    "documentdate": "invoice_date",
    "InvoiceDate": "invoice_date",
    "invoicedate": "invoice_date",
    "invoice_date": "invoice_date",
    "GrossAmount": "total_amount",
    "grossamount": "total_amount",
    "TotalAmount": "total_amount",
    "totalamount": "total_amount",
    "total_amount": "total_amount",
    "NetAmount": "total_amount",
    "Currency": "_currency",
    "currency": "_currency",
}


def _flatten_invoice_output(parsed: dict | list) -> dict:
    """Normalize the (possibly nested) invoice model output into a flat dict."""
    flat: dict[str, str] = {}
    items = parsed if isinstance(parsed, list) else [parsed]
    for item in items:
        if not isinstance(item, dict):
            continue
        for raw_key, raw_val in item.items():
            if isinstance(raw_val, list):
                for sub in raw_val:
                    if isinstance(sub, dict):
                        for k, v in sub.items():
                            mapped = _INV_FIELD_MAP.get(k)
                            if mapped and v:
                                flat.setdefault(mapped, str(v).strip())
            elif isinstance(raw_val, dict):
                for k, v in raw_val.items():
                    mapped = _INV_FIELD_MAP.get(k)
                    if mapped and v:
                        flat.setdefault(mapped, str(v).strip())
            else:
                mapped = _INV_FIELD_MAP.get(raw_key)
                if mapped and raw_val:
                    flat.setdefault(mapped, str(raw_val).strip())
    return flat


def donut_extract_fields(image_path: str | Path) -> dict:
    """
    Unified extraction: invoice model + DocVQA fallback.
    Returns a dict with the same 6 keys as the regex extractor.
    """
    image = Image.open(image_path).convert("RGB")

    inv_raw = extract_with_invoice_model(image)
    inv_fields = _flatten_invoice_output(inv_raw)

    # Append currency to total_amount if present
    currency = inv_fields.pop("_currency", None)
    if currency and "total_amount" in inv_fields:
        inv_fields["total_amount"] = f"{inv_fields['total_amount']} {currency}"

    # DocVQA questions for fields the invoice model doesn't cover
    VQA_QUESTIONS = {
        "issuer_name": "Who is the seller or issuer of this invoice?",
        "recipient_name": "Who is the buyer or recipient of this invoice?",
        "due_date": "What is the due date of this invoice?",
    }

    vqa_fields: dict[str, str | None] = {}
    for field, question in VQA_QUESTIONS.items():
        vqa_fields[field] = extract_with_docvqa(image, question)

    TARGET_FIELDS = [
        "invoice_number",
        "invoice_date",
        "due_date",
        "issuer_name",
        "recipient_name",
        "total_amount",
    ]
    result: dict[str, str | None] = {}
    for f in TARGET_FIELDS:
        val = inv_fields.get(f) or vqa_fields.get(f)
        result[f] = val if val else None

    return result


print("Extraction functions defined.")

## 4) Single-image test

In [ ]:
test_images = sorted(IMAGES_DIR.glob("Template7_Instance0.*"))
if not test_images:
    test_images = sorted(IMAGES_DIR.glob("*Instance0.*"))

if test_images:
    test_path = test_images[0]
    print(f"Test image: {test_path.name}")
    result = donut_extract_fields(test_path)
    for k, v in result.items():
        print(f"  {k:20s}: {v}")
else:
    print("No images found in", IMAGES_DIR)

## 5) Batch extraction on `images/` sample

In [ ]:
IMAGES_PER_TEMPLATE = 2

by_tpl: dict[int, list] = defaultdict(list)
for p in sorted(IMAGES_DIR.iterdir()):
    if not p.is_file():
        continue
    if p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}:
        continue
    m = re.match(r"Template(\d+)_Instance(\d+)", p.stem, re.I)
    if not m:
        continue
    by_tpl[int(m.group(1))].append((int(m.group(2)), p))

images_sample = []
for tid in sorted(by_tpl.keys()):
    paths = [pt for _, pt in sorted(by_tpl[tid], key=lambda x: x[0])][:IMAGES_PER_TEMPLATE]
    images_sample.extend(paths)

print(f"Templates: {len(by_tpl)}, sample size: {len(images_sample)}")

results_donut = []
errors_donut = []
t0 = time.time()

for idx, path in enumerate(images_sample, start=1):
    try:
        row = donut_extract_fields(path)
        m = re.match(r"Template(\d+)_", path.name, re.I)
        row["template_id"] = int(m.group(1)) if m else None
        row["file_name"] = path.name
        results_donut.append(row)
    except Exception as exc:
        errors_donut.append({"file_name": path.name, "error": str(exc)})
    if idx % 5 == 0 or idx == len(images_sample):
        elapsed = time.time() - t0
        print(f"  [{idx}/{len(images_sample)}] elapsed {elapsed:.1f}s")

print(f"\nDone. Extracted: {len(results_donut)}, errors: {len(errors_donut)}")
if results_donut:
    print("\nFirst result:")
    for k, v in results_donut[0].items():
        print(f"  {k:20s}: {v}")

## 6) Save results to CSV

In [ ]:
RESULTS_CSV = PROJECT_ROOT / "images_donut_extraction_results.csv"
ERRORS_CSV = PROJECT_ROOT / "images_donut_extraction_errors.csv"

CSV_FIELDS = [
    "template_id",
    "file_name",
    "invoice_number",
    "invoice_date",
    "due_date",
    "issuer_name",
    "recipient_name",
    "total_amount",
]

if results_donut:
    with RESULTS_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS, extrasaction="ignore")
        w.writeheader()
        for r in results_donut:
            w.writerow({k: r.get(k) for k in CSV_FIELDS})
    print(f"Saved {len(results_donut)} rows to {RESULTS_CSV.name}")
else:
    print("No results to save.")

if errors_donut:
    with ERRORS_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["file_name", "error"])
        w.writeheader()
        w.writerows(errors_donut)
    print(f"Saved {len(errors_donut)} errors to {ERRORS_CSV.name}")

## 7) Compare with regex extraction results

Load both the Donut and regex CSVs, then compute per-field agreement rates.

In [ ]:
REGEX_CSV = PROJECT_ROOT / "images_ai_extraction_results.csv"

def _load_csv(path: Path) -> dict[str, dict]:
    """Load CSV into {file_name: {field: value}} lookup."""
    rows: dict[str, dict] = {}
    if not path.exists():
        return rows
    with path.open(encoding="utf-8") as f:
        for r in csv.DictReader(f):
            rows[r["file_name"]] = r
    return rows


def _fuzzy_eq(a: str | None, b: str | None) -> bool:
    """Case-insensitive, whitespace-normalized equality."""
    if a is None and b is None:
        return True
    if a is None or b is None:
        return False
    return " ".join(a.lower().split()) == " ".join(b.lower().split())


def _contains_match(a: str | None, b: str | None) -> bool:
    """Check if one value is a substring of the other (case-insensitive)."""
    if a is None or b is None:
        return False
    al, bl = a.lower().strip(), b.lower().strip()
    return al in bl or bl in al


donut_rows = _load_csv(RESULTS_CSV)
regex_rows = _load_csv(REGEX_CSV)

if not regex_rows:
    print(f"Regex results CSV not found at {REGEX_CSV.name}.")
    print("Run the regex extraction notebook first, then re-run this cell.")
else:
    common_files = sorted(set(donut_rows) & set(regex_rows))
    print(f"Comparing {len(common_files)} files present in both CSVs\n")

    COMPARE_FIELDS = [
        "invoice_number", "invoice_date", "due_date",
        "issuer_name", "recipient_name", "total_amount",
    ]

    header = f"{'Field':20s} {'Exact':>7s} {'Fuzzy':>7s} {'Substr':>7s} {'Both None':>10s} {'Total':>6s}"
    print(header)
    print("-" * len(header))

    for field in COMPARE_FIELDS:
        exact = fuzzy = substr = both_none = 0
        for fn in common_files:
            d_val = donut_rows[fn].get(field) or None
            r_val = regex_rows[fn].get(field) or None
            if d_val == r_val:
                exact += 1
            if _fuzzy_eq(d_val, r_val):
                fuzzy += 1
            if _contains_match(d_val, r_val):
                substr += 1
            if d_val is None and r_val is None:
                both_none += 1
        n = len(common_files)
        print(
            f"{field:20s} {exact:6d} ({exact*100//n:2d}%)"
            f" {fuzzy:5d} ({fuzzy*100//n:2d}%)"
            f" {substr:5d} ({substr*100//n:2d}%)"
            f" {both_none:9d}"
            f" {n:5d}"
        )

    print("\nSample side-by-side (first 5 common files):")
    for fn in common_files[:5]:
        print(f"\n── {fn} ──")
        for field in COMPARE_FIELDS:
            d = donut_rows[fn].get(field, "")
            r = regex_rows[fn].get(field, "")
            flag = "✓" if _fuzzy_eq(d, r) else "✗"
            print(f"  {flag} {field:20s}  donut={d!r:30s}  regex={r!r}")